# Gradio Day!

Today we will build User Interfaces using the outrageously simple Gradio framework.

In [15]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [16]:
import gradio as gr

In [17]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [18]:
openai = OpenAI()

system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message}, 
        {"role": "user", "content": prompt}
    ]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

In [19]:
message_gpt("What is today's date?")

APIConnectionError: Connection error.

## User Interface time!

In [20]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [21]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("abhay", "itsasecret"))

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# And now - changing the function from "shout" to "message_gpt"

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Hello", "Howdy"],
    flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Let's use Markdown
system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your messages:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
    ],
    flagging_mode="never"
)

view.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Let's create a call that streams back results

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result # lazy calling, this helps to print it one character at a time

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


## And now getting fancy

Remember to check the Intermediate Python Guide if you're unsure about generators and "yield"

In [ ]:
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [ ]:
# The ! tells certain notebook environments, such as Jupyter Notebook or Google Colab, to execute the following text as a shell command rather than as Python code.
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
Error: pull model manifest: Get "https://web-notification.capgemini.com?url=https%3a%2f%2fregistry%2eollama%2eai%2fv2%2flibrary%2fllama3%2e2%2fmanifests%2flatest&referer=&reason=Not+allowed+to+browse+Generative+AI+and+ML+Applications+category&reasoncode=CATEGORY_DENIED&timebound=1&action=deny&kind=category&rule=1508972&cat=Generative+AI+and+ML+Applications&user=abhay.shrivastava@capgemini.com&locid=0ab1e196&lang=en_US&zsq=KNRNqbDfLvZT7SqjbK4kRvLvJnf47r6bD0tFZkNzsq": blocked redirect to a different host


In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

def stream_llama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_model(prompt, model):
    if model=="GPT_Mini-4.1":
        result = stream_gpt(prompt)
    elif model=="Llama3.2":
        result = stream_llama(prompt)
    else:
        raise ValueError("Unknown Model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT_Mini-4.1", "Llama3.2"], label="Select model", value="GPT_Mini-4.1")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT_Mini-4.1"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Llama3.2"]
        ], 
    flagging_mode="never"
    )
view.launch(inbrowser=True, auth=["abhay", "secret"])

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


# Building a company brochure generator

Now you know how - it's simple!

In [ ]:
from scraper import fetch_website_contents

In [ ]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    
    if model=="GPT_Mini-4.1":
        result = stream_gpt(prompt)
    elif model=="Llama3.2":
        result = stream_llama(prompt)
    else:
        raise ValueError("Unknown Model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT_Mini-4.1", "Llama3.2"], label="Select model", value="GPT_Mini-4.1")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT_Mini-4.1"],
        ["Edward Donner", "https://edwarddonner.com", "Llama3.2"]
    ],
    flagging_mode="never"
)

view.launch(inbrowser=True, auth=["abhay", "secret"])

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\abshriva\OneDrive - Capgemini\Desktop\Projects\LLM-Engineering\.venv\Lib\site-packages\urllib3\connectionpool.py", line 470, in _make_request
    self._validate_conn(conn)
  File "c:\Users\abshriva\OneDrive - Capgemini\Desktop\Projects\LLM-Engineering\.venv\Lib\site-packages\urllib3\connectionpool.py", line 1125, in _validate_conn
    conn.connect()
  File "c:\Users\abshriva\OneDrive - Capgemini\Desktop\Projects\LLM-Engineering\.venv\Lib\site-packages\urllib3\connection.py", line 878, in connect
    wrapped_socket, is_verified = _ssl_wrap_socket_and_match_hostname(
                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\abshriva\OneDrive - Capgemini\Desktop\Projects\LLM-Engineering\.venv\Lib\site-packages\urllib3\connection.py", line 1080, in _ssl_wrap_socket_and_match_hostname
    ssl_sock = ssl_wrap_socket(
               ^^^^^^^^^^^^^^^^
  File "c:\Users\abshriva\OneDrive - Capgemini\Desktop\Projects